# 6_CLUSTERING — QRF Mixture-of-Experts Uncertainty Quantification

Implements the clustering-based Quantile Regression Forest (QRF) Mixture-of-Experts
framework from Al-Aghbary et al. (2026, GJI) applied to Antarctic and Greenland
subglacial heat-flow prediction.

| Section | Description |
|---------|-------------|
| 0 | Toggles & config |
| 1 | Load & clean reference data |
| **1b** | **Coarsen target grids 5 km → 25 km (5×5 mean)** |
| 2 | K-Means cluster selection (NCLUSTERS 2–5) |
| 3 | Cluster diagnostics (PCA, violins, centroids) |
| 4 | Apply clusters to coarsened Antarctic & Greenland grids |
| 5 | QRF implementation (Meinshausen-style wrapper) |
| 6 | Per-cluster Expert + Baseline QRF training |
| 7 | 5-fold cross-validation — CV threshold derivation |
| 8 | Diagnostic field computation helper functions |
| 9 | Apply MoE to target grids → write NetCDF |
| 10 | Diagnostic maps Antarctica & Greenland |
| 11 | Supplementary figures |
| 12 | Save artefacts & summary |

**Prerequisites**: `1_Import.ipynb` and `4x_QRF.ipynb` have been run.  
Required files: `data/IHFCobs.parquet`, `data/antarctica.parquet`,
`data/greenland.parquet`, `output/models/qrf_artefacts.pkl`.


## Section 0 — Toggles & Config

In [ ]:
# Execution toggles
RUN_COARSEN       = True   # False = load saved 25-km parquets
RUN_CLUSTERING    = True   # False = load saved k-means from output/clustering
RUN_QRF_TRAINING  = True   # False = load saved expert artefacts
RUN_CV            = True   # False = load saved CV thresholds
RUN_GRID_APPLY    = True   # False = load saved target-grid NetCDFs

# Cluster sweep range (NOT the K used elsewhere in this project)
NCLUSTERS_RANGE = range(2, 6)          # test NCLUSTERS = 2, 3, 4, 5
NCLUSTERS_BEST  = None                 # set to int to override auto-selection

# ── Coarsening ────────────────────────────────────────────────────────────────
# Native grid spacing (m) from config.py
NATIVE_SPACING_ANT = 5_000   # 5 km (EPSG:3031)
NATIVE_SPACING_GRL = 5_000   # 5 km (EPSG:3413)
TARGET_SPACING_M   = 25_000  # 25 km
COARSEN_FACTOR     = TARGET_SPACING_M // NATIVE_SPACING_ANT  # = 5

# ── Imports ──────────────────────────────────────────────────────────────────
import sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from pathlib import Path
from copy import deepcopy

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
import netCDF4 as nc

sys.path.insert(0, ".")
from config import (
    obs_model, parquet_ref, parquet_ant, parquet_grl,
    ant_Aq2_qrf_nc, grl_Kq2_qrf_nc,
    model_paths, sweep_dir,
    q_clip_min, q_clip_max,
    random_state, N_CV_FOLDS,
    QRF_N_ESTIMATORS, QRF_MAX_FEATURES, QRF_MIN_SAMPLES_LEAF, QRF_MAX_DEPTH, QRF_N_JOBS,
    ENTROPY_BINS, FIGDPI, FIGEXT,
    hfc_map, unc_cmap,
    output_root, model_dir,
    milli,
)

warnings.filterwarnings("ignore")

# Output directories
CLUSTER_DIR = Path("output/clustering")
FIG_DIR     = Path("fig/6_CLUSTERING")
CRS_ANT25   = Path("data/antarctica_25km.parquet")
CRS_GRL25   = Path("data/greenland_25km.parquet")
for d in (CLUSTER_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Helpers
def to_mW(x):
    """Convert W/m² array to mW/m² for display."""
    return np.asarray(x) * milli

print(f"obs_model features: {len(obs_model)}")
print(f"NCLUSTERS sweep: {list(NCLUSTERS_RANGE)}")
print(f"Coarsen factor:  {COARSEN_FACTOR}x  "
      f"({NATIVE_SPACING_ANT/1e3:.0f} km -> {TARGET_SPACING_M/1e3:.0f} km)")


## Section 1 — Load & Clean Reference Data

In [ ]:
df_ref_raw = pd.read_parquet(parquet_ref)
print(f"Raw reference rows: {len(df_ref_raw):,}")

# Load QRF artefacts to get obssel (sweep-derived feature subset)
qrf_bundle = pickle.load(open(model_paths["qrf"], "rb"))
obssel     = qrf_bundle["obssel"]
print(f"obssel: {len(obssel)} features")

# Filter valid heat-flow range AND complete obssel features
cols_needed = obssel + ["q"]
df_ref = df_ref_raw.dropna(subset=cols_needed).copy()
df_ref = df_ref[(df_ref["q"] >= q_clip_min) & (df_ref["q"] <= q_clip_max)].reset_index(drop=True)
print(f"After NaN/clip filter: {len(df_ref):,} rows")

X_train = df_ref[obssel].values.astype(np.float64)
y_train = df_ref["q"].values.astype(np.float64)
print(f"Training matrix shape: {X_train.shape}")

# ── Clustering feature matrix (obs_model, median-imputed) ─────────────────
cluster_cols = obs_model   # full feature set for clustering

df_cluster_ref = df_ref_raw.dropna(subset=["q"]).copy()
df_cluster_ref = df_cluster_ref[
    (df_cluster_ref["q"] >= q_clip_min) & (df_cluster_ref["q"] <= q_clip_max)
].reset_index(drop=True)

# Column medians from reference set (used for target-grid imputation later)
cluster_medians = df_cluster_ref[cluster_cols].median()

X_clust_ref    = (df_cluster_ref[cluster_cols]
                  .fillna(cluster_medians)
                  .values.astype(np.float64))
clust_scaler   = StandardScaler().fit(X_clust_ref)
X_clust_ref_sc = clust_scaler.transform(X_clust_ref)

print(f"Clustering matrix shape: {X_clust_ref_sc.shape}")
print(f"NaN after imputation: {np.isnan(X_clust_ref_sc).sum()}")


## Section 1b — Coarsen Target Grids 5 km → 25 km

Each 25 km super-cell is the **nanmean** of all valid 5 km cells within
its 5 × 5 block (vectorised via pandas `groupby` → `mean`).  
Cells where all features are NaN after aggregation are retained and will be
median-imputed before clustering (same as the native grid).

Coarsened parquets are written to `data/antarctica_25km.parquet` and
`data/greenland_25km.parquet`; reload with `RUN_COARSEN = False`.


In [ ]:
def coarsen_grid(parquet_path: Path,
                 native_m: float,
                 target_m: float,
                 out_path: Path) -> pd.DataFrame:
    """
    Coarsen a regular projected-grid parquet from native_m to target_m.

    Strategy
    --------
    1. Assign each point to a 25-km super-cell bin via floor-division of (x, y).
    2. Aggregate ALL numeric columns with nanmean in one vectorised groupby.
    3. Super-cell centre coordinates = mean x / mean y of member cells.

    Parameters
    ----------
    parquet_path : Path to native-resolution parquet.
    native_m     : Native grid spacing in metres (5 000).
    target_m     : Target grid spacing in metres (25 000).
    out_path     : Destination parquet path.

    Returns
    -------
    pd.DataFrame of the coarsened grid.
    """
    factor = int(round(target_m / native_m))
    assert factor >= 1, f"target_m must be >= native_m (factor={factor})"

    df = pd.read_parquet(parquet_path)
    print(f"  Native rows  : {len(df):>10,}")

    # Bin keys: floor-divide projected coordinate by target spacing
    half      = native_m / 2.0
    df["_bx"] = np.floor((df["x"] + half) / target_m).astype(np.int32)
    df["_by"] = np.floor((df["y"] + half) / target_m).astype(np.int32)

    # Keep only numeric columns (features + x/y); drop bin keys from result
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    agg_cols  = [c for c in num_cols if c not in ("_bx", "_by")]

    # One-pass vectorised aggregation
    agg = (df.groupby(["_bx", "_by"], sort=False)[agg_cols]
             .mean()            # pandas mean skips NaN by default
             .reset_index(drop=True))

    print(f"  Coarsened    : {len(agg):>10,}  (factor {factor}x)")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    agg.to_parquet(out_path, index=False)
    print(f"  Saved -> {out_path}")
    return agg


if RUN_COARSEN:
    print("Antarctica ->")
    df_ant25 = coarsen_grid(parquet_ant, NATIVE_SPACING_ANT, TARGET_SPACING_M, CRS_ANT25)
    print("Greenland ->")
    df_grl25 = coarsen_grid(parquet_grl, NATIVE_SPACING_GRL, TARGET_SPACING_M, CRS_GRL25)
else:
    df_ant25 = pd.read_parquet(CRS_ANT25)
    df_grl25 = pd.read_parquet(CRS_GRL25)
    print(f"Loaded coarsened grids:  ANT {len(df_ant25):,}   GRL {len(df_grl25):,}")

print(f"\nCoarsened grid sizes:")
print(f"  Antarctica 25 km : {len(df_ant25):,} cells")
print(f"  Greenland  25 km : {len(df_grl25):,} cells")


## Section 2 — K-Means Cluster Selection

Sweep NCLUSTERS 2–5 using Elbow (WCSS) + Davies-Bouldin score.  
Auto-selects best N by minimum Davies-Bouldin (ties broken by elbow inflection).


In [ ]:
kmeans_models = {}   # NCLUSTERS -> fitted KMeans
ref_labels    = {}   # NCLUSTERS -> label array len(df_cluster_ref)
diag_rows     = []

if RUN_CLUSTERING:
    for ncval in NCLUSTERS_RANGE:
        print(f"Fitting KMeans N={ncval} ...", end=" ", flush=True)
        km = KMeans(n_clusters=ncval, random_state=random_state,
                    n_init=20, max_iter=500)
        km.fit(X_clust_ref_sc)
        kmeans_models[ncval] = km
        ref_labels[ncval]    = km.labels_

        sil = silhouette_score(X_clust_ref_sc, km.labels_,
                               sample_size=min(10_000, len(X_clust_ref_sc)),
                               random_state=random_state)
        db  = davies_bouldin_score(X_clust_ref_sc, km.labels_)
        sizes = pd.Series(km.labels_).value_counts().sort_index().tolist()
        diag_rows.append(dict(NCLUSTERS=ncval,
                              Inertia=round(km.inertia_, 1),
                              Silhouette=round(sil, 4),
                              Davies_Bouldin=round(db, 4),
                              Cluster_sizes=sizes))
        pkl_path = CLUSTER_DIR / f"kmeans_nc{ncval}.pkl"
        pickle.dump(km, open(pkl_path, "wb"))
        print(f"done.  Inertia={km.inertia_:.1f}  DB={db:.4f}  Sil={sil:.4f}")
else:
    for ncval in NCLUSTERS_RANGE:
        pkl_path = CLUSTER_DIR / f"kmeans_nc{ncval}.pkl"
        km = pickle.load(open(pkl_path, "rb"))
        kmeans_models[ncval] = km
        ref_labels[ncval]    = km.predict(X_clust_ref_sc)
        sil = silhouette_score(X_clust_ref_sc, ref_labels[ncval],
                               sample_size=min(10_000, len(X_clust_ref_sc)),
                               random_state=random_state)
        db  = davies_bouldin_score(X_clust_ref_sc, ref_labels[ncval])
        sizes = pd.Series(ref_labels[ncval]).value_counts().sort_index().tolist()
        diag_rows.append(dict(NCLUSTERS=ncval,
                              Inertia=round(km.inertia_, 1),
                              Silhouette=round(sil, 4),
                              Davies_Bouldin=round(db, 4),
                              Cluster_sizes=sizes))
        print(f"Loaded KMeans N={ncval} from {pkl_path}")

diag_df = pd.DataFrame(diag_rows)

if NCLUSTERS_BEST is None:
    best_idx       = diag_df["Davies_Bouldin"].idxmin()
    NCLUSTERS_BEST = int(diag_df.loc[best_idx, "NCLUSTERS"])
    print(f"\nAuto-selected NCLUSTERS_BEST = {NCLUSTERS_BEST}  (minimum Davies-Bouldin)")

print(diag_df.to_string(index=False))


## Section 3 — Cluster Diagnostics (PCA, violins, centroids)

In [ ]:
best_labels = ref_labels[NCLUSTERS_BEST]
best_km     = kmeans_models[NCLUSTERS_BEST]

# Elbow + Davies-Bouldin + Silhouette panel
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ks = diag_df["NCLUSTERS"].values

axes[0].plot(ks, diag_df["Inertia"].values, "o-")
axes[0].axvline(NCLUSTERS_BEST, color="red", ls="--", label=f"K={NCLUSTERS_BEST}")
axes[0].set_xlabel("K"); axes[0].set_ylabel("WCSS (inertia)")
axes[0].set_title("Elbow"); axes[0].legend()

axes[1].plot(ks, diag_df["Davies_Bouldin"].values, "o-", color="darkorange")
axes[1].axvline(NCLUSTERS_BEST, color="red", ls="--")
axes[1].set_xlabel("K"); axes[1].set_ylabel("Davies-Bouldin")
axes[1].set_title("Davies-Bouldin (lower = better)")

axes[2].plot(ks, diag_df["Silhouette"].values, "o-", color="steelblue")
axes[2].axvline(NCLUSTERS_BEST, color="red", ls="--")
axes[2].set_xlabel("K"); axes[2].set_ylabel("Silhouette")
axes[2].set_title("Silhouette (higher = better)")

fig.tight_layout()
out = FIG_DIR / f"S_cluster_selection{FIGEXT}"
fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
print(f"Saved {out}")

# PCA 2-D scatter
pca  = PCA(n_components=2, random_state=random_state)
Xpca = pca.fit_transform(X_clust_ref_sc)
ev   = pca.explained_variance_ratio_ * 100
cmap_disc = plt.get_cmap("Set1", NCLUSTERS_BEST)

fig, ax = plt.subplots(figsize=(6, 5))
for ci in range(NCLUSTERS_BEST):
    mask = best_labels == ci
    ax.scatter(Xpca[mask, 0], Xpca[mask, 1],
               s=3, alpha=0.4, color=cmap_disc(ci), label=f"Cluster {ci}")
ax.set_xlabel(f"PC1 {ev[0]:.1f}%"); ax.set_ylabel(f"PC2 {ev[1]:.1f}%")
ax.set_title(f"PCA cluster separation  N={NCLUSTERS_BEST}")
ax.legend(markerscale=3, loc="best")
fig.tight_layout()
out = FIG_DIR / f"S_cluster_pca{FIGEXT}"
fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
print(f"Saved {out}")

# Centroid heatmap
centres_phys = clust_scaler.inverse_transform(best_km.cluster_centers_)
df_centres   = pd.DataFrame(centres_phys, columns=cluster_cols)
df_norm      = (df_centres - df_centres.min()) / (df_centres.max() - df_centres.min() + 1e-12)

fig, ax = plt.subplots(figsize=(16, 3))
im = ax.imshow(df_norm.values, aspect="auto", cmap="RdYlBu_r", vmin=0, vmax=1)
ax.set_xticks(range(len(cluster_cols)))
ax.set_xticklabels(cluster_cols, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(NCLUSTERS_BEST))
ax.set_yticklabels([f"Cluster {i}" for i in range(NCLUSTERS_BEST)])
ax.set_title(f"Centroid profiles  N={NCLUSTERS_BEST} — normalised physical units")
plt.colorbar(im, ax=ax, label="Normalised value")
fig.tight_layout()
out = FIG_DIR / f"S_cluster_centroids{FIGEXT}"
fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
print(f"Saved {out}")

diag_df.to_csv(CLUSTER_DIR / "cluster_diagnostics.csv", index=False)
print("Saved cluster_diagnostics.csv")


## Section 4 — Apply Cluster Labels to Coarsened Target Grids

NaN cells in target grids are filled with training-set column medians before
clustering so every subglacial grid cell receives a valid cluster label.
The imputation fills only for clustering; QRF inference handles NaNs separately.


In [ ]:
DOMAINS = {"ant": CRS_ANT25, "grl": CRS_GRL25}
target_labels = {}   # domain -> label array

for domain, pq_path in DOMAINS.items():
    print(f"\n{domain.upper()}")
    df_tgt = pd.read_parquet(pq_path)
    print(f"  Grid rows: {len(df_tgt):,}")

    missing = [f for f in cluster_cols if f not in df_tgt.columns]
    if missing:
        print(f"  Missing features (will impute): {missing}")
        for m in missing:
            df_tgt[m] = np.nan

    X_tgt = (df_tgt[cluster_cols]
             .replace([np.inf, -np.inf], np.nan)
             .fillna(cluster_medians)
             .values.astype(np.float64))

    X_tgt_sc = clust_scaler.transform(X_tgt)
    lbl = kmeans_models[NCLUSTERS_BEST].predict(X_tgt_sc).astype(np.int16)
    target_labels[domain] = lbl

    counts = pd.Series(lbl).value_counts().sort_index().to_dict()
    print(f"  Cluster sizes N={NCLUSTERS_BEST}: {counts}")

    coord_cols = [c for c in ("lon", "lat", "x", "y") if c in df_tgt.columns]
    out_df = df_tgt[coord_cols].copy()
    out_df[f"cluster_n{NCLUSTERS_BEST}"] = lbl
    out_path = CLUSTER_DIR / f"{domain}_cluster_labels.parquet"
    out_df.to_parquet(out_path, index=False)
    print(f"  Saved {out_path}")


## Section 5 — QRF Implementation (Meinshausen 2006 wrapper)

Extends `sklearn.RandomForestRegressor` to:
- Retain all training response values in each terminal node
- Estimate quantiles empirically from retained leaf values
- Decompose predictive variance into aleatoric (within-leaf) and epistemic (between-tree)

Reference: Meinshausen N. (2006). Quantile Regression Forests. *JMLR* 7:983-999.


In [ ]:
class QuantileRegressionForest(RandomForestRegressor):
    """
    Meinshausen (2006) Quantile Regression Forest.
    Wraps sklearn RandomForestRegressor; retains all leaf response values
    for empirical quantile estimation and aleatoric/epistemic decomposition.
    """

    def fit(self, X, y, sample_weight=None):
        super().fit(X, y, sample_weight=sample_weight)
        leaf_ids = self.apply(X)          # (n_samples, n_estimators)
        self._leaf_values = []
        for t in range(self.n_estimators):
            lv = {}
            for node, yval in zip(leaf_ids[:, t], y):
                lv.setdefault(node, []).append(yval)
            self._leaf_values.append(lv)
        return self

    def predict_mean(self, X):
        """Empirical leaf mean, vectorised over estimators."""
        lids  = self.apply(X)
        preds = np.array([
            [np.mean(self._leaf_values[t].get(lids[i, t], [0.0]))
             for t in range(self.n_estimators)]
            for i in range(len(X))
        ])
        return preds.mean(axis=1)

    def predict_quantile(self, X, q):
        """Empirical quantile q in (0,1) from pooled leaf distributions."""
        lids = self.apply(X)
        out  = np.empty(len(X))
        for i in range(len(X)):
            pooled = np.concatenate(
                [self._leaf_values[t].get(lids[i, t], [0.0])
                 for t in range(self.n_estimators)]
            )
            out[i] = np.quantile(pooled, q)
        return out

    def variance_decomposition(self, X):
        """Returns (aleatoric, epistemic) variance arrays.
        Aleatoric = mean within-leaf variance; Epistemic = variance of tree means."""
        lids  = self.apply(X)
        n, T  = len(X), self.n_estimators
        mu_t  = np.zeros((n, T))
        var_t = np.zeros((n, T))
        for t in range(T):
            for i in range(n):
                vals       = np.asarray(self._leaf_values[t].get(lids[i, t], [0.0]))
                mu_t[i, t] = vals.mean()
                var_t[i, t]= vals.var()
        return var_t.mean(axis=1), mu_t.var(axis=1)

    def robustness(self, X, n_bins=50):
        """R_i = 1 - H_n(i) / log2(n_bins),  R in [0,1].
        H_n is Shannon entropy of the discretised predictive CDF."""
        lids = self.apply(X)
        out  = np.empty(len(X))
        for i in range(len(X)):
            pooled = np.concatenate(
                [self._leaf_values[t].get(lids[i, t], [0.0])
                 for t in range(self.n_estimators)]
            )
            counts, _ = np.histogram(pooled, bins=n_bins)
            probs = counts / (counts.sum() + 1e-12)
            probs = probs[probs > 0]
            H     = -np.sum(probs * np.log2(probs))
            out[i]= 1.0 - H / np.log2(n_bins)
        return np.clip(out, 0.0, 1.0)


print("QuantileRegressionForest class defined.")


## Section 6 — Per-Cluster Expert + Baseline QRF Training

Trains:
1. **Baseline QRF** — global model on all training data (no clustering)
2. **Expert QRF[k]** — one QRF per cluster, trained only on cluster members

Hyperparameters loaded from sweep JSON if available, else config defaults.


In [ ]:
qrf_param_path = sweep_dir / "qrf_params.json"
if qrf_param_path.exists():
    qrf_params = json.loads(qrf_param_path.read_text())
    NTREES  = int(qrf_params.get("n_estimators", QRF_N_ESTIMATORS))
    MAXFEAT = qrf_params.get("max_features",  QRF_MAX_FEATURES)
    MINLEAF = int(qrf_params.get("min_samples_leaf", max(QRF_MIN_SAMPLES_LEAF, 3)))
    print(f"Loaded QRF params from {qrf_param_path}")
else:
    NTREES  = QRF_N_ESTIMATORS
    MAXFEAT = QRF_MAX_FEATURES
    MINLEAF = max(QRF_MIN_SAMPLES_LEAF, 3)
    print("Using config defaults for QRF hyperparameters.")
print(f"NTREES={NTREES}  MAXFEAT={MAXFEAT}  MINLEAF={MINLEAF}")

# Map training rows -> cluster labels
X_ref_for_clust = (df_ref[cluster_cols]
                   .replace([np.inf, -np.inf], np.nan)
                   .fillna(cluster_medians)
                   .values.astype(np.float64))
X_ref_clust_sc       = clust_scaler.transform(X_ref_for_clust)
train_cluster_labels = kmeans_models[NCLUSTERS_BEST].predict(X_ref_clust_sc).astype(int)
df_ref = df_ref.copy()
df_ref["cluster"] = train_cluster_labels
print(f"Training cluster sizes: {pd.Series(train_cluster_labels).value_counts().sort_index().to_dict()}")

expert_models  = {}
baseline_model = None

if RUN_QRF_TRAINING:
    n_base  = len(X_train)
    md_base = max(5, round(0.9 * np.log2(n_base)))
    print(f"Fitting Baseline QRF  n={n_base}  max_depth={md_base} ...", end=" ", flush=True)
    baseline_model = QuantileRegressionForest(
        n_estimators=NTREES, max_features=MAXFEAT,
        min_samples_leaf=MINLEAF, max_depth=md_base,
        n_jobs=QRF_N_JOBS, random_state=random_state,
    ).fit(X_train, y_train)
    print("done.")

    for ci in range(NCLUSTERS_BEST):
        mask = train_cluster_labels == ci
        Xc, yc = X_train[mask], y_train[mask]
        nc  = len(yc)
        mdc = max(5, round(0.9 * np.log2(nc)))
        print(f"Fitting Expert {ci}  n={nc}  max_depth={mdc} ...", end=" ", flush=True)
        expert_models[ci] = QuantileRegressionForest(
            n_estimators=NTREES, max_features=MAXFEAT,
            min_samples_leaf=MINLEAF, max_depth=mdc,
            n_jobs=QRF_N_JOBS, random_state=random_state,
        ).fit(Xc, yc)
        print("done.")

    artefact_path = CLUSTER_DIR / f"expert_qrf_n{NCLUSTERS_BEST}.pkl"
    pickle.dump(
        dict(baseline=baseline_model, experts=expert_models,
             obssel=obssel, cluster_cols=cluster_cols,
             kmeans=kmeans_models[NCLUSTERS_BEST],
             clust_scaler=clust_scaler,
             cluster_medians=cluster_medians,
             NCLUSTERS_BEST=NCLUSTERS_BEST),
        open(artefact_path, "wb"),
    )
    print(f"\nSaved expert artefacts -> {artefact_path}")
else:
    artefact_path = CLUSTER_DIR / f"expert_qrf_n{NCLUSTERS_BEST}.pkl"
    bundle = pickle.load(open(artefact_path, "rb"))
    baseline_model = bundle["baseline"]
    expert_models  = bundle["experts"]
    print(f"Loaded expert artefacts from {artefact_path}")


## Section 7 — 5-Fold Cross-Validation (CV Threshold Derivation)

Runs stratified K-fold CV to derive data-driven thresholds for the
Confidence and Explainability maps:
- `b_mean_cv` — mean bandwidth (semi-IQR) on held-out folds
- `b_excess_cv`, `b_deficit_cv` — excess/deficit bandwidth thresholds
- `Rcv25`, `Rcv75` — 25th/75th-percentile robustness on held-out folds
- `A50_cv`, `E50_cv` — median aleatoric / epistemic variance on held-out folds


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

if RUN_CV:
    skf          = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=random_state)
    strat_labels = train_cluster_labels

    oof_base = dict(mean=[], q25=[], q75=[], aleat=[], epist=[], robust=[], ytrue=[])
    oof_moe  = dict(mean=[], q25=[], q75=[], aleat=[], epist=[], robust=[], ytrue=[])

    print(f"Running {N_CV_FOLDS}-fold CV ...")
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_train, strat_labels)):
        print(f"  Fold {fold_i+1}/{N_CV_FOLDS}", end=" ", flush=True)

        Xtr, ytr   = X_train[train_idx], y_train[train_idx]
        Xval, yval = X_train[val_idx],   y_train[val_idx]
        cl_tr  = strat_labels[train_idx]
        cl_val = strat_labels[val_idx]

        # Baseline fold
        n_b    = len(ytr)
        md_b   = max(5, round(0.9 * np.log2(n_b)))
        base_f = QuantileRegressionForest(
            n_estimators=NTREES, max_features=MAXFEAT,
            min_samples_leaf=MINLEAF, max_depth=md_b,
            n_jobs=QRF_N_JOBS, random_state=random_state,
        ).fit(Xtr, ytr)
        oof_base["mean"].append(base_f.predict_mean(Xval))
        oof_base["q25"].append(base_f.predict_quantile(Xval, 0.25))
        oof_base["q75"].append(base_f.predict_quantile(Xval, 0.75))
        al_b, ep_b = base_f.variance_decomposition(Xval)
        oof_base["aleat"].append(al_b)
        oof_base["epist"].append(ep_b)
        oof_base["robust"].append(base_f.robustness(Xval, n_bins=ENTROPY_BINS))
        oof_base["ytrue"].append(yval)

        # MoE fold
        experts_f = {}
        for ci in np.unique(cl_tr):
            m = cl_tr == ci
            nc_  = m.sum()
            mdc_ = max(5, round(0.9 * np.log2(nc_)))
            experts_f[ci] = QuantileRegressionForest(
                n_estimators=NTREES, max_features=MAXFEAT,
                min_samples_leaf=MINLEAF, max_depth=mdc_,
                n_jobs=QRF_N_JOBS, random_state=random_state,
            ).fit(Xtr[m], ytr[m])

        moe_pred = {k: np.full(len(yval), np.nan)
                    for k in ("mean", "q25", "q75", "aleat", "epist", "robust")}
        for ci, expf in experts_f.items():
            m = cl_val == ci
            if not m.any():
                continue
            moe_pred["mean"][m]   = expf.predict_mean(Xval[m])
            moe_pred["q25"][m]    = expf.predict_quantile(Xval[m], 0.25)
            moe_pred["q75"][m]    = expf.predict_quantile(Xval[m], 0.75)
            al_m, ep_m            = expf.variance_decomposition(Xval[m])
            moe_pred["aleat"][m]  = al_m
            moe_pred["epist"][m]  = ep_m
            moe_pred["robust"][m] = expf.robustness(Xval[m], n_bins=ENTROPY_BINS)

        for k in ("mean", "q25", "q75", "aleat", "epist", "robust"):
            oof_moe[k].append(moe_pred[k])
        oof_moe["ytrue"].append(yval)
        print("done.")

    for k in oof_moe:
        oof_moe[k]  = np.concatenate(oof_moe[k])
        oof_base[k] = np.concatenate(oof_base[k])

    bw_cv        = 0.5 * (oof_moe["q75"] - oof_moe["q25"])
    b_mean_cv    = float(np.nanmean(bw_cv))
    b_excess_cv  = float(np.nanmean(bw_cv[bw_cv >= b_mean_cv]))
    b_deficit_cv = float(np.nanmean(bw_cv[bw_cv <  b_mean_cv]))
    Rcv25        = float(np.nanquantile(oof_moe["robust"], 0.25))
    Rcv75        = float(np.nanquantile(oof_moe["robust"], 0.75))
    A50_cv       = float(np.nanmedian(oof_moe["aleat"]))
    E50_cv       = float(np.nanmedian(oof_moe["epist"]))

    thresholds = dict(
        b_mean_cv=b_mean_cv, b_excess_cv=b_excess_cv, b_deficit_cv=b_deficit_cv,
        Rcv25=Rcv25, Rcv75=Rcv75, A50_cv=A50_cv, E50_cv=E50_cv,
    )
    thresh_path = CLUSTER_DIR / "cv_thresholds.json"
    with open(thresh_path, "w") as fp:
        json.dump({k: float(v) for k, v in thresholds.items()}, fp, indent=2)
    print(f"CV complete.  Thresholds saved -> {thresh_path}")
    for k, v in thresholds.items():
        suf = " mW/m2" if "b_" in k else ""
        print(f"  {k:20s}: {to_mW(v):.5f}{suf}" if "b_" in k else f"  {k:20s}: {v:.5f}")

else:
    thresholds   = json.loads((CLUSTER_DIR / "cv_thresholds.json").read_text())
    b_mean_cv    = thresholds["b_mean_cv"]
    b_excess_cv  = thresholds["b_excess_cv"]
    b_deficit_cv = thresholds["b_deficit_cv"]
    Rcv25        = thresholds["Rcv25"]
    Rcv75        = thresholds["Rcv75"]
    A50_cv       = thresholds["A50_cv"]
    E50_cv       = thresholds["E50_cv"]
    print("Loaded CV thresholds from file.")


## Section 8 — Diagnostic Field Computation Helper Functions

Implements the five diagnostic fields for any set of QRF predictions:
1. Bandwidth `b = 0.5*(Q0.75 - Q0.25)` (eq. 5)
2. Variance `sigma2_t = sigma2_a + sigma2_e` (eq. 1)
3. Robustness `R = 1 - H_n` (eqs A1-A2)
4. Confidence — 5-category map (eqs 6-10)
5. Explainability — 4-category map (eqs 11-14)


In [ ]:
# Confidence category codes (eqs 6-10)
CONF_HIGH     = 0  # b <= b_mean  AND  R >= Rcv75
CONF_MODERATE = 1  # b <= b_mean  AND  Rcv25 <= R < Rcv75
CONF_LOW      = 2  # b <= b_mean  AND  R < Rcv25
CONF_OVER     = 3  # b < b_deficit  AND  R < Rcv25  (overconfident)
CONF_UNDER    = 4  # b > b_excess   AND  R >= Rcv75 (underconfident)
CONF_LABELS   = ["High", "Moderate", "Low", "Overconfident", "Underconfident"]
CONF_COLORS   = ["#2ca25f", "#99d8c9", "#fc8d59", "#d7191c", "#ffd700"]

# Explainability category codes (eqs 11-14)
EXPL_LOW   = 0  # a <= A50  AND  e <= E50  (low uncertainty)
EXPL_EPIST = 1  # a <= A50  AND  e >  E50  (epistemic-dominated)
EXPL_ALEAT = 2  # a >  A50  AND  e <= E50  (aleatoric-dominated)
EXPL_HIGH  = 3  # a >  A50  AND  e >  E50  (high uncertainty)
EXPL_LABELS= ["Low uncertainty", "Epistemic-dominated",
              "Aleatoric-dominated", "High uncertainty"]
EXPL_COLORS= ["#2166ac", "#abd9e9", "#fdae61", "#d73027"]


def compute_diagnostics(model_or_dict, X, cluster_labels, thresholds,
                        n_bins=ENTROPY_BINS):
    """
    Compute all five diagnostic fields for prediction inputs X.

    Parameters
    ----------
    model_or_dict  : QuantileRegressionForest | dict{cluster_id: QRF}
    X              : np.ndarray  (n_samples, n_features)
    cluster_labels : np.ndarray  (n_samples,) int
    thresholds     : dict from CV
    n_bins         : int — entropy bins for robustness

    Returns dict with keys:
        mean, q25, q75, bandwidth, variance_total,
        variance_aleatoric, variance_epistemic,
        robustness, confidence (int8), explainability (int8)
    """
    n   = X.shape[0]
    out = {k: np.full(n, np.nan) for k in
           ("mean", "q25", "q75", "bandwidth", "variance_total",
            "variance_aleatoric", "variance_epistemic", "robustness")}
    out["confidence"]     = np.full(n, -1, dtype=np.int8)
    out["explainability"] = np.full(n, -1, dtype=np.int8)

    models_by_cluster = (
        model_or_dict if isinstance(model_or_dict, dict)
        else {ci: model_or_dict for ci in np.unique(cluster_labels)}
    )

    for ci, model in models_by_cluster.items():
        mask = cluster_labels == ci
        if not mask.any():
            continue
        Xc = X[mask]
        out["mean"][mask]                = model.predict_mean(Xc)
        out["q25"][mask]                 = model.predict_quantile(Xc, 0.25)
        out["q75"][mask]                 = model.predict_quantile(Xc, 0.75)
        al, ep                           = model.variance_decomposition(Xc)
        out["variance_aleatoric"][mask]  = al
        out["variance_epistemic"][mask]  = ep
        out["robustness"][mask]          = model.robustness(Xc, n_bins=n_bins)

    out["bandwidth"]      = 0.5 * (out["q75"] - out["q25"])
    out["variance_total"] = out["variance_aleatoric"] + out["variance_epistemic"]

    # Confidence map
    b  = out["bandwidth"]
    R  = out["robustness"]
    bm  = thresholds["b_mean_cv"]
    bex = thresholds["b_excess_cv"]
    bdf = thresholds["b_deficit_cv"]
    r25 = thresholds["Rcv25"]
    r75 = thresholds["Rcv75"]
    conf = out["confidence"]
    conf[(b <= bm)  & (R >= r75)]              = CONF_HIGH
    conf[(b <= bm)  & (R >= r25) & (R < r75)] = CONF_MODERATE
    conf[(b <= bm)  & (R <  r25)]              = CONF_LOW
    conf[(b <  bdf) & (R <  r25)]              = CONF_OVER
    conf[(b >  bex) & (R >= r75)]              = CONF_UNDER
    conf[conf == -1]                           = CONF_MODERATE  # catch-all

    # Explainability map
    sa  = out["variance_aleatoric"]
    se  = out["variance_epistemic"]
    A50 = thresholds["A50_cv"]
    E50 = thresholds["E50_cv"]
    expl = out["explainability"]
    expl[(sa <= A50) & (se <= E50)] = EXPL_LOW
    expl[(sa <= A50) & (se >  E50)] = EXPL_EPIST
    expl[(sa >  A50) & (se <= E50)] = EXPL_ALEAT
    expl[(sa >  A50) & (se >  E50)] = EXPL_HIGH

    return out


print("compute_diagnostics helper defined.")


## Section 9 — Apply MoE to Target Grids → Write NetCDF

For each domain (Antarctica, Greenland):
1. Load cluster labels from Section 4
2. Load `obssel` features from the 25-km parquet grid
3. Route each grid cell to its cluster Expert
4. Compute all 5 diagnostic fields
5. Append new variables to the existing QRF NetCDF (creates if absent)


In [ ]:
DOMAIN_CONFIG = {
    "ant": dict(parquet=CRS_ANT25, nc_path=ant_Aq2_qrf_nc, xcol="x", ycol="y", epsg=3031),
    "grl": dict(parquet=CRS_GRL25, nc_path=grl_Kq2_qrf_nc, xcol="x", ycol="y", epsg=3413),
}

obssel_medians = pd.Series(np.nanmedian(X_train, axis=0), index=obssel)
grid_diag = {}   # domain -> diagnostics dict

if RUN_GRID_APPLY:
    for domain, cfg in DOMAIN_CONFIG.items():
        print(f"\n{domain.upper()}")
        df_tgt  = pd.read_parquet(cfg["parquet"])
        lbl_path= CLUSTER_DIR / f"{domain}_cluster_labels.parquet"
        df_lbl  = pd.read_parquet(lbl_path)
        cl_col  = f"cluster_n{NCLUSTERS_BEST}"
        cluster_lbl = df_lbl[cl_col].values.astype(int)

        missing_sel = [f for f in obssel if f not in df_tgt.columns]
        for m in missing_sel:
            df_tgt[m] = np.nan

        X_tgt = (df_tgt[obssel]
                 .replace([np.inf, -np.inf], np.nan)
                 .fillna(obssel_medians)
                 .values.astype(np.float64))
        print(f"  Grid points: {len(X_tgt):,}")

        diag = compute_diagnostics(expert_models, X_tgt, cluster_lbl, thresholds)
        grid_diag[domain] = diag
        print(f"  Diagnostics computed.")
        print(f"  Mean GHF: {to_mW(np.nanmean(diag['mean'])):.1f} mW/m2")

        nc_path = cfg["nc_path"]
        if not Path(nc_path).exists():
            print(f"  WARNING: NetCDF not found at {nc_path} — skipping NC write.")
            print(f"  Diagnostics available in grid_diag['{domain}'] dict.")
            continue

        with nc.Dataset(nc_path, "a") as ds:
            new_vars = {
                "qrf_moe_mean":           ("GHF MoE ensemble mean",          "W m-2",  diag["mean"]),
                "qrf_moe_q25":            ("GHF MoE Q0.25",                  "W m-2",  diag["q25"]),
                "qrf_moe_q75":            ("GHF MoE Q0.75",                  "W m-2",  diag["q75"]),
                "qrf_moe_bandwidth":      ("Prediction interval semi-IQR",   "W m-2",  diag["bandwidth"]),
                "qrf_moe_variance_total": ("Total predictive variance",       "W2 m-4", diag["variance_total"]),
                "qrf_moe_variance_aleat": ("Aleatoric variance",              "W2 m-4", diag["variance_aleatoric"]),
                "qrf_moe_variance_epist": ("Epistemic variance",              "W2 m-4", diag["variance_epistemic"]),
                "qrf_moe_robustness":     ("Robustness 1-Hn",                "1",      diag["robustness"]),
                "qrf_moe_confidence":     ("Confidence 0=High 4=Underconf",  "1",      diag["confidence"].astype(np.float32)),
                "qrf_moe_explainability": ("Explainability 0=Low 3=High",    "1",      diag["explainability"].astype(np.float32)),
            }
            dim = list(ds.variables.keys())[0]
            for vname, (long_name, units, data) in new_vars.items():
                if vname in ds.variables:
                    ds.variables[vname][:] = data
                else:
                    v = ds.createVariable(vname, "f4", ds.variables[dim].dimensions)
                    v.long_name = long_name
                    v.units     = units
                    v[:]        = data
            ds.qrf_moe_N_clusters = NCLUSTERS_BEST
            ds.qrf_moe_obssel     = str(obssel)
        print(f"  NetCDF updated -> {nc_path}")

    for domain, diag in grid_diag.items():
        pickle.dump(diag, open(CLUSTER_DIR / f"{domain}_diagnostics.pkl", "wb"))
    print("\nDiagnostic fields saved.")

else:
    for domain in DOMAIN_CONFIG:
        diag_path = CLUSTER_DIR / f"{domain}_diagnostics.pkl"
        grid_diag[domain] = pickle.load(open(diag_path, "rb"))
    print("Loaded grid diagnostics from saved files.")


## Section 10 — Diagnostic Maps Antarctica & Greenland

Publication figures equivalent to Figs 9-10 of Al-Aghbary et al. (2026):
- **Fig 9 equivalent** — Robustness + Total Variance (2-panel)
- **Fig 10 equivalent** — Confidence (categorical) + Explainability (categorical, 2-panel)

Maps use `imshow` on the 25-km projected grid.
All variance quantities converted to mW2/m4 for display.


In [ ]:
def grid_extent(df, xcol, ycol):
    return df[xcol].min(), df[xcol].max(), df[ycol].min(), df[ycol].max()


def pivot_field(df, xcol, ycol, values):
    """Pivot flat 1-D field onto 2-D grid for imshow."""
    df2 = df[[xcol, ycol]].copy()
    df2["v"] = values
    pivot = df2.pivot_table(index=ycol, columns=xcol, values="v", aggfunc="mean")
    return pivot.values[::-1], pivot.columns.values, pivot.index.values[::-1]


for domain, cfg in DOMAIN_CONFIG.items():
    print(f"Maps -- {domain.upper()}")
    diag   = grid_diag[domain]
    df_tgt = pd.read_parquet(cfg["parquet"])
    xc, yc = cfg["xcol"], cfg["ycol"]
    km     = 1 / 1000

    # Fig 9 equivalent: Robustness + Total Variance
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    fig.suptitle(f"{domain.upper()} -- Robustness & Total Variance  MoE QRF  (25 km)", fontsize=13)

    rob_grid, xs, ys = pivot_field(df_tgt, xc, yc, diag["robustness"])
    im0 = axes[0].imshow(rob_grid, origin="upper",
                         extent=[xs.min()*km, xs.max()*km, ys.min()*km, ys.max()*km],
                         cmap="viridis", vmin=0, vmax=1, aspect="equal")
    plt.colorbar(im0, ax=axes[0], label="Robustness [0-1]", fraction=0.046, pad=0.04)
    axes[0].set_title("(a) Robustness")
    axes[0].set_xlabel("x  km"); axes[0].set_ylabel("y  km")

    var_mW2 = to_mW(to_mW(diag["variance_total"]))   # W2/m4 -> mW2/m4
    vmax_var = np.nanpercentile(var_mW2, 98)
    var_grid, xs, ys = pivot_field(df_tgt, xc, yc, var_mW2)
    im1 = axes[1].imshow(var_grid, origin="upper",
                         extent=[xs.min()*km, xs.max()*km, ys.min()*km, ys.max()*km],
                         cmap=unc_cmap, vmin=0, vmax=vmax_var, aspect="equal")
    plt.colorbar(im1, ax=axes[1], label="Total variance  mW2 m-4", fraction=0.046, pad=0.04)
    axes[1].set_title("(b) Total variance")
    axes[1].set_xlabel("x  km")

    fig.tight_layout()
    out = FIG_DIR / f"{domain}_fig9_robustness_variance{FIGEXT}"
    fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
    print(f"  Saved {out}")

    # Fig 10 equivalent: Confidence + Explainability
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle(f"{domain.upper()} -- Confidence & Explainability  MoE QRF  (25 km)", fontsize=13)

    cmap_conf = mcolors.ListedColormap(CONF_COLORS)
    bounds_c  = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
    norm_c    = mcolors.BoundaryNorm(bounds_c, cmap_conf.N)
    conf_grid, xs, ys = pivot_field(df_tgt, xc, yc, diag["confidence"].astype(float))
    im2 = axes[0].imshow(conf_grid, origin="upper",
                         extent=[xs.min()*km, xs.max()*km, ys.min()*km, ys.max()*km],
                         cmap=cmap_conf, norm=norm_c, aspect="equal", interpolation="none")
    patches_c = [mpatches.Patch(color=CONF_COLORS[i], label=CONF_LABELS[i])
                 for i in range(len(CONF_LABELS))]
    axes[0].legend(handles=patches_c, loc="lower left", fontsize=7, framealpha=0.8)
    axes[0].set_title("(a) Confidence"); axes[0].set_xlabel("x  km"); axes[0].set_ylabel("y  km")

    cmap_expl = mcolors.ListedColormap(EXPL_COLORS)
    bounds_e  = [-0.5, 0.5, 1.5, 2.5, 3.5]
    norm_e    = mcolors.BoundaryNorm(bounds_e, cmap_expl.N)
    expl_grid, xs, ys = pivot_field(df_tgt, xc, yc, diag["explainability"].astype(float))
    im3 = axes[1].imshow(expl_grid, origin="upper",
                         extent=[xs.min()*km, xs.max()*km, ys.min()*km, ys.max()*km],
                         cmap=cmap_expl, norm=norm_e, aspect="equal", interpolation="none")
    patches_e = [mpatches.Patch(color=EXPL_COLORS[i], label=EXPL_LABELS[i])
                 for i in range(len(EXPL_LABELS))]
    axes[1].legend(handles=patches_e, loc="lower left", fontsize=7, framealpha=0.8)
    axes[1].set_title("(b) Explainability"); axes[1].set_xlabel("x  km")

    fig.tight_layout()
    out = FIG_DIR / f"{domain}_fig10_confidence_explainability{FIGEXT}"
    fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
    print(f"  Saved {out}")


## Section 11 — Supplementary Figures

In [ ]:
# S1 -- Aleatoric / Epistemic variance decomposition maps
for domain, cfg in DOMAIN_CONFIG.items():
    diag   = grid_diag[domain]
    df_tgt = pd.read_parquet(cfg["parquet"])
    xc, yc = cfg["xcol"], cfg["ycol"]
    km     = 1 / 1000

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    fig.suptitle(f"{domain.upper()} -- Variance Decomposition  (25 km)", fontsize=13)
    for ax, key, title in zip(axes,
                              ("variance_aleatoric", "variance_epistemic"),
                              ("(a) Aleatoric variance", "(b) Epistemic variance")):
        v_mW2    = to_mW(to_mW(diag[key]))
        g, xs, ys = pivot_field(df_tgt, xc, yc, v_mW2)
        vmax      = np.nanpercentile(v_mW2, 98)
        im = ax.imshow(g, origin="upper",
                       extent=[xs.min()*km, xs.max()*km, ys.min()*km, ys.max()*km],
                       cmap=unc_cmap, vmin=0, vmax=vmax, aspect="equal")
        plt.colorbar(im, ax=ax, label="Variance  mW2 m-4", fraction=0.046, pad=0.04)
        ax.set_title(title); ax.set_xlabel("x  km")
    axes[0].set_ylabel("y  km")
    fig.tight_layout()
    out = FIG_DIR / f"S_{domain}_variance_decomposition{FIGEXT}"
    fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
    print(f"Saved {out}")

# S2 -- Cluster spatial maps on the coarsened target grids
cmap_disc2 = plt.get_cmap("Set1", NCLUSTERS_BEST)
for domain, cfg in DOMAIN_CONFIG.items():
    df_tgt = pd.read_parquet(cfg["parquet"])
    xc, yc = cfg["xcol"], cfg["ycol"]
    km     = 1 / 1000
    lbl_path = CLUSTER_DIR / f"{domain}_cluster_labels.parquet"
    df_lbl   = pd.read_parquet(lbl_path)
    cl_col   = f"cluster_n{NCLUSTERS_BEST}"

    g, xs, ys = pivot_field(df_tgt, xc, yc, df_lbl[cl_col].astype(float).values)
    fig, ax   = plt.subplots(figsize=(5.5, 5.5))
    cmap_c    = mcolors.ListedColormap([cmap_disc2(i) for i in range(NCLUSTERS_BEST)])
    bounds    = np.arange(-0.5, NCLUSTERS_BEST + 0.5)
    norm      = mcolors.BoundaryNorm(bounds, cmap_c.N)
    ax.imshow(g, origin="upper",
              extent=[xs.min()*km, xs.max()*km, ys.min()*km, ys.max()*km],
              cmap=cmap_c, norm=norm, aspect="equal", interpolation="none")
    patches = [mpatches.Patch(color=cmap_disc2(i), label=f"Cluster {i}")
               for i in range(NCLUSTERS_BEST)]
    ax.legend(handles=patches, loc="lower left", fontsize=8)
    ax.set_title(f"{domain.upper()} -- Cluster map  N={NCLUSTERS_BEST}  (25 km)")
    ax.set_xlabel("x  km"); ax.set_ylabel("y  km")
    fig.tight_layout()
    out = FIG_DIR / f"S_{domain}_cluster_map{FIGEXT}"
    fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
    print(f"Saved {out}")

# S3 -- CV uncertainty decomposition violin: MoE vs Baseline (only if CV was run)
if RUN_CV and "oof_moe" in dir():
    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    fig.suptitle("CV uncertainty decomposition -- Baseline vs MoE", fontsize=12)
    keys_units = [("aleat", "mW2 m-4"), ("epist", "mW2 m-4"),
                  ("robust", "Robustness")]
    for ax, (key, unit), title in zip(
        axes, keys_units,
        ["(a) Aleatoric variance", "(b) Epistemic variance", "(c) Robustness"]
    ):
        bv = to_mW(to_mW(oof_base[key])) if key != "robust" else oof_base[key]
        mv = to_mW(to_mW(oof_moe[key]))  if key != "robust" else oof_moe[key]
        ax.violinplot([bv, mv], positions=[1, 2], showmedians=True, showextrema=False)
        ax.set_xticks([1, 2]); ax.set_xticklabels(["Baseline", "MoE"])
        ax.set_ylabel(unit); ax.set_title(title)
    fig.tight_layout()
    out = FIG_DIR / f"S_cv_uncertainty_decomposition{FIGEXT}"
    fig.savefig(out, dpi=FIGDPI, bbox_inches="tight"); plt.show()
    print(f"Saved {out}")


## Section 12 — Save Artefacts & Summary

In [ ]:
summary = dict(
    model_version    = "03",
    NCLUSTERS_BEST   = NCLUSTERS_BEST,
    coarsen_factor   = COARSEN_FACTOR,
    native_spacing_m = NATIVE_SPACING_ANT,
    target_spacing_m = TARGET_SPACING_M,
    n_features_sel   = len(obssel),
    n_features_clust = len(cluster_cols),
    n_train          = int(len(df_ref)),
    n_cv_folds       = N_CV_FOLDS,
    cv_thresholds    = {k: float(v) for k, v in thresholds.items()},
    diagnostics      = diag_df.to_dict(orient="records"),
    outputs = {
        "cluster_diagnostics_csv"  : str(CLUSTER_DIR / "cluster_diagnostics.csv"),
        "cv_thresholds_json"       : str(CLUSTER_DIR / "cv_thresholds.json"),
        "expert_artefacts_pkl"     : str(CLUSTER_DIR / f"expert_qrf_n{NCLUSTERS_BEST}.pkl"),
        "ant_cluster_labels"       : str(CLUSTER_DIR / "ant_cluster_labels.parquet"),
        "grl_cluster_labels"       : str(CLUSTER_DIR / "grl_cluster_labels.parquet"),
        "ant_diagnostics_pkl"      : str(CLUSTER_DIR / "ant_diagnostics.pkl"),
        "grl_diagnostics_pkl"      : str(CLUSTER_DIR / "grl_diagnostics.pkl"),
        "ant_25km_parquet"         : str(CRS_ANT25),
        "grl_25km_parquet"         : str(CRS_GRL25),
    },
)

out_json = CLUSTER_DIR / "cluster_summary.json"
with open(out_json, "w") as fp:
    json.dump(summary, fp, indent=2)
print(f"Saved summary -> {out_json}")

print("\nOutput checklist:")
for label, path in summary["outputs"].items():
    exists = Path(path).exists()
    print(f"  {'ok' if exists else 'MISSING'}  {label:35s}  {path}")
